> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🧑‍🏫 수행평가 채점 도우미 (교사용)

학생용 노트북(10_assessment_project)과 **같은 함수**로 조건을 만들고 결과를 다시 계산해요. 자세한 채점 기준과 운영 방법은 docs/ASSESSMENT.md에 있어요.

1. **명단과 조건표** — 우리 반 명단을 넣으면 학생마다의 SEED·TEST_SIZE·꼭 할 실험·확인 행이 나와요.
2. **제출 노트북 한꺼번에 점검** — 학생들이 낸 .ipynb를 한 폴더에 모으면, 요약코드로 점수를 **다시 계산**하고 명단·꼭 할 실험·빈 답·**비슷한 답**·답 속 숫자 근거를 표 하나(점검표.csv)로 보여 줘요. ⚠️ 표시가 있는 학생만 자세히 보면 돼요.
3. **현장 확인 설명지와 답 열쇠** — 학생마다 다른 행을 고른 설명지(현장설명지.md)와 교사용 답 열쇠(현장답열쇠.md)를 만들어요.
4. **점수 합계** — 노트북 수준과 현장 확인(○·△·×)을 적으면, 현장 확인으로 상한을 둔 최종 점수와 채점표 CSV가 나와요.

- 명단을 붙여 넣고 실행하면 학번마다의 나만의 조건이 나와요. 확인행은 학생 화면과 같이 "테스트 데이터 맨 위에서 몇 번째 행"이에요.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import hashlib, json

실험과제_목록 = [
    '입력 속성 하나를 빼고 다시 학습해, 그 속성이 성능에 얼마나 도움이 되는지 비교하기',
    '하이퍼파라미터(knn은 k, tree는 max_depth)를 세 가지 값으로 바꿔 성능 비교하기',
    '정규화(0~1로 크기 맞추기)를 하기 전과 한 뒤의 성능 비교하기',
    '훈련 데이터를 절반만 써서 학습해 성능 비교하기',
    '다른 알고리즘 하나로 바꿔 성능 비교하기',
]

def 나만의_조건(학번):
    h = int(hashlib.sha256(('ai-project:' + str(학번).strip()).encode('utf-8')).hexdigest(), 16)
    return {
        'SEED': h % 10000,
        'TEST_SIZE': [0.2, 0.25, 0.3][(h // 1000) % 3],
        '실험과제': 실험과제_목록[(h // 3000) % len(실험과제_목록)],
        '확인행': (h // 15000) % 5,   # 테스트 데이터에서 0부터 센 자리 — 화면에는 맨 위에서 몇 번째인지(+1)로 보여요
    }

def 지문_만들기(요약):
    글 = json.dumps(요약, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(글.encode('utf-8')).hexdigest()[:8]


import pandas as pd
import seaborn as sns
from sklearn import datasets as skd

def _사이킷런(이름):
    return getattr(skd, 이름)(as_frame=True).frame

def _씨본(이름):
    return sns.load_dataset(이름).drop(columns=['Unnamed: 0'], errors='ignore')

데이터_목록 = {
    'penguins': dict(분야='생명과학·환경', 설명='남극 펭귄 344마리의 몸 측정값', 불러오기=lambda: _씨본('penguins'),
        TARGET='species', FEATURES=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], 예시='종 분류 · 몸무게 회귀',
        열={'species': '종(Adelie·Chinstrap·Gentoo)', 'island': '사는 섬', 'bill_length_mm': '부리길이(mm)', 'bill_depth_mm': '부리깊이(mm)',
           'flipper_length_mm': '날개길이(mm)', 'body_mass_g': '몸무게(g)', 'sex': '성별'}),
    'iris': dict(분야='생명과학·농업', 설명='붓꽃 150송이의 꽃잎·꽃받침 크기', 불러오기=lambda: _씨본('iris'),
        TARGET='species', FEATURES=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'], 예시='품종 분류',
        열={'sepal_length': '꽃받침 길이(cm)', 'sepal_width': '꽃받침 너비(cm)', 'petal_length': '꽃잎 길이(cm)', 'petal_width': '꽃잎 너비(cm)', 'species': '품종'}),
    'wine': dict(분야='화학·식품', 설명='와인 178병의 화학 성분 13가지와 품종', 불러오기=lambda: _사이킷런('load_wine'),
        TARGET='target', FEATURES=['alcohol', 'flavanoids', 'color_intensity', 'proline'], 예시='품종(0·1·2) 분류',
        열={'target': '품종(0·1·2)', 'alcohol': '알코올 도수', 'malic_acid': '사과산', 'flavanoids': '플라보노이드',
           'color_intensity': '색의 진하기', 'hue': '색조', 'proline': '프롤린(아미노산)', 'total_phenols': '페놀 총량'}),
    'breast_cancer': dict(분야='의학·보건', 설명='유방 종양 569개의 세포핵 측정값 30가지와 진단', 불러오기=lambda: _사이킷런('load_breast_cancer'),
        TARGET='target', FEATURES=['mean radius', 'mean texture', 'mean smoothness', 'mean concave points'], 예시='악성(0)·양성(1) 분류',
        열={'target': '진단(0 악성 · 1 양성)', 'mean radius': '세포핵 반지름 평균', 'mean texture': '표면 질감(명암 차이) 평균',
           'mean smoothness': '테두리 매끄러움 평균', 'mean concave points': '테두리의 오목한 곳 수 평균'}),
    'diabetes': dict(분야='의학·약학·간호', 설명='당뇨 환자 442명의 검사값과 1년 뒤 병의 진행 정도', 불러오기=lambda: _사이킷런('load_diabetes'),
        TARGET='target', FEATURES=['bmi', 'bp', 's5', 'age'], 예시='진행 정도(숫자) 회귀',
        열={'target': '1년 뒤 병의 진행 정도(클수록 나빠짐)', 'age': '나이(크기를 맞춘 값)', 'sex': '성별(크기를 맞춘 값)',
           'bmi': '체질량지수(크기를 맞춘 값)', 'bp': '평균 혈압(크기를 맞춘 값)', 's1': '혈액 검사 1 (총콜레스테롤)', 's5': '혈액 검사 5 (중성지방 관련)'}),
    'exercise': dict(분야='체육·운동과학', 설명='식단·운동 종류·잰 때에 따른 맥박 90번', 불러오기=lambda: _씨본('exercise'),
        TARGET='pulse', FEATURES=['diet', 'kind', 'time'], 예시='맥박 회귀 · 운동 종류 분류',
        열={'id': '참가자 번호', 'diet': '식단(low fat · no fat)', 'pulse': '맥박(1분에 뛴 수)', 'time': '잰 때(운동 1·15·30분)', 'kind': '운동(rest · walking · running)'}),
    'attention': dict(분야='심리·교육', 설명='주의 집중 방식에 따른 문제 풀이 점수 60번', 불러오기=lambda: _씨본('attention'), 목적='회귀',
        TARGET='score', FEATURES=['attention', 'solutions'], 예시='점수 회귀',
        열={'subject': '참가자 번호', 'attention': '집중 방식(divided 나눠서 · focused 한 곳에)', 'solutions': '풀이 방법 수', 'score': '점수'}),
    'tips': dict(분야='경영·서비스', 설명='식당 계산서 244장과 손님이 낸 팁', 불러오기=lambda: _씨본('tips'),
        TARGET='tip', FEATURES=['total_bill', 'size', 'time', 'smoker'], 예시='팁 금액 회귀 · 점심/저녁 분류',
        열={'total_bill': '계산서 금액(달러)', 'tip': '팁(달러)', 'sex': '계산한 사람 성별', 'smoker': '흡연석 여부',
           'day': '요일', 'time': '점심·저녁', 'size': '일행 수'}),
    'diamonds': dict(분야='경제·디자인', 설명='다이아몬드 53,940개의 크기·품질과 가격', 불러오기=lambda: _씨본('diamonds'),
        TARGET='price', FEATURES=['carat', 'cut', 'color', 'clarity'], 예시='가격 회귀 · 커팅 등급 분류',
        열={'carat': '무게(캐럿)', 'cut': '커팅 등급', 'color': '색 등급(D가 가장 좋음)', 'clarity': '투명도 등급',
           'depth': '깊이 비율(%)', 'table': '윗면 비율(%)', 'price': '가격(달러)', 'x': '길이(mm)', 'y': '너비(mm)', 'z': '높이(mm)'}),
    'california': dict(분야='부동산·도시 계획', 설명='캘리포니아 20,640개 구역의 집값과 동네 정보', 불러오기=lambda: _사이킷런('fetch_california_housing'),
        TARGET='MedHouseVal', FEATURES=['MedInc', 'HouseAge', 'AveRooms', 'Latitude', 'Longitude'], 예시='집값 회귀',
        열={'MedInc': '구역 가구 소득 중앙값(만 달러)', 'HouseAge': '집 나이 중앙값(년)', 'AveRooms': '가구당 평균 방 수', 'AveBedrms': '가구당 평균 침실 수',
           'Population': '구역 인구', 'AveOccup': '가구당 평균 사람 수', 'Latitude': '위도', 'Longitude': '경도', 'MedHouseVal': '집값 중앙값(10만 달러)'}),
    'mpg': dict(분야='기계·자동차 공학', 설명='자동차 398대의 엔진·무게와 연비', 불러오기=lambda: _씨본('mpg'),
        TARGET='mpg', FEATURES=['weight', 'horsepower', 'cylinders', 'model_year'], 예시='연비 회귀 · 생산 지역 분류',
        열={'mpg': '연비(마일/갤런)', 'cylinders': '실린더 수', 'displacement': '배기량', 'horsepower': '마력', 'weight': '무게(파운드)',
           'acceleration': '0→60마일 가속 시간(초)', 'model_year': '연식(70 = 1970년)', 'origin': '생산 지역', 'name': '차 이름'}),
    'taxis': dict(분야='교통·도시', 설명='뉴욕 택시 6,433번의 운행 기록', 불러오기=lambda: _씨본('taxis'),
        TARGET='fare', FEATURES=['distance', 'passengers', 'pickup_borough'], 예시='요금 회귀 · 결제 수단 분류',
        열={'passengers': '승객 수', 'distance': '거리(마일)', 'fare': '요금(달러)', 'tip': '팁', 'tolls': '통행료', 'total': '총액',
           'color': '택시 색', 'payment': '결제 수단', 'pickup_borough': '탄 구역', 'dropoff_borough': '내린 구역'}),
    'flights': dict(분야='항공·관광', 설명='1949~1960년 달마다 항공 승객 수', 불러오기=lambda: _씨본('flights'),
        TARGET='passengers', FEATURES=['year', 'month'], 예시='승객 수 회귀',
        열={'year': '연도', 'month': '달', 'passengers': '승객 수(천 명)'}),
    'titanic': dict(분야='역사·사회', 설명='타이타닉호 승객 891명의 정보와 생존 여부', 불러오기=lambda: _씨본('titanic'),
        TARGET='survived', FEATURES=['pclass', 'sex', 'age', 'fare'], 예시='생존(0·1) 분류',
        열={'survived': '생존(1)·사망(0)', 'pclass': '객실 등급(1·2·3)', 'sex': '성별', 'age': '나이', 'sibsp': '함께 탄 형제·배우자 수',
           'parch': '함께 탄 부모·자녀 수', 'fare': '요금', 'embarked': '탄 항구'}),
    'car_crashes': dict(분야='법·사회 안전', 설명='미국 51개 주의 교통사고와 그 원인', 불러오기=lambda: _씨본('car_crashes'),
        TARGET='total', FEATURES=['speeding', 'alcohol', 'not_distracted', 'no_previous'], 예시='사고율 회귀',
        열={'total': '운전자 10억 마일당 사고 사망 운전자 수', 'speeding': '그중 과속', 'alcohol': '그중 음주',
           'not_distracted': '그중 한눈팔지 않음', 'no_previous': '그중 사고 이력 없음', 'ins_premium': '보험료', 'ins_losses': '보험금 지급'}),
    'healthexp': dict(분야='보건 정책·국제', 설명='6개 나라의 해마다 의료비와 기대수명', 불러오기=lambda: _씨본('healthexp'),
        TARGET='Life_Expectancy', FEATURES=['Spending_USD', 'Year', 'Country'], 예시='기대수명 회귀',
        열={'Year': '연도', 'Country': '나라', 'Spending_USD': '1인당 의료비(달러)', 'Life_Expectancy': '기대수명(세)'}),
    'geyser': dict(분야='지구과학', 설명='옐로스톤 간헐천의 분출 272번', 불러오기=lambda: _씨본('geyser'),
        TARGET='waiting', FEATURES=['duration', 'kind'], 예시='기다린 시간 회귀 · 분출 종류 분류',
        열={'duration': '분출 시간(분)', 'waiting': '분출 전에 기다린 시간(분)', 'kind': '분출 종류(long·short)'}),
    'planets': dict(분야='천문·우주', 설명='지금까지 발견된 외계 행성 1,035개', 불러오기=lambda: _씨본('planets'),
        TARGET='method', FEATURES=['orbital_period', 'distance', 'year', 'number'], 예시='발견 방법 분류',
        열={'method': '발견 방법', 'number': '그 별에 있는 행성 수', 'orbital_period': '공전 주기(일)',
           'mass': '질량(목성 = 1)', 'distance': '지구에서의 거리(파섹)', 'year': '발견 연도'}),
    'digits': dict(분야='컴퓨터·인공지능', 설명='8×8 손글씨 숫자 그림 1,797장', 불러오기=lambda: _사이킷런('load_digits'),
        TARGET='target', FEATURES='pixel', 예시='숫자(0~9) 분류',
        열={'target': '쓴 숫자(0~9)', 'pixel_0_0 ~ pixel_7_7': '8×8 칸마다의 진하기(0~16)'}),
}

def 데이터_불러오기(코드, my_csv=''):
    if 코드 == 'my_csv':
        return pd.read_csv(my_csv)
    return 데이터_목록[코드]['불러오기']()


import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

def 학습_목적(y_raw, 직접=None):
    if 직접 in ('분류', '회귀'):
        return 직접
    글자 = not pd.api.types.is_numeric_dtype(y_raw) or pd.api.types.is_bool_dtype(y_raw)
    return '분류' if 글자 or y_raw.nunique() <= 10 else '회귀'

def 숫자_열(df, features):
    return [c for c in features if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]

def 울타리(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)

def 전처리(df, features, target, 목적, missing='평균', outlier='그대로'):
    로그 = {'처음': len(df)}
    d = df.loc[~df.duplicated()]                        # 모든 열이 똑같은 행만 겹친 행으로 봐요
    로그['겹친 행'] = len(df) - len(d)
    d = d[features + [target]].copy()
    n = len(d); d = d.dropna(subset=[target]); 로그['정답이 빈 행'] = n - len(d)
    n = len(d); d = d.dropna(subset=features, how='all'); 로그['입력이 모두 빈 행'] = n - len(d)
    숫자 = 숫자_열(d, features)
    글자 = [c for c in features if c not in 숫자]
    if outlier == '지우기':
        n = len(d)
        밖 = pd.Series(False, index=d.index)
        건너뜀 = []
        for c in 숫자:
            낮은, 높은 = 울타리(df[c])                   # 2-1과 같은 울타리(처음 데이터로 한 번만 구해요)
            if 낮은 == 높은:
                건너뜀.append(c)                        # 값이 거의 다 같아 울타리가 0이면 건너뛰어요
                continue
            밖 |= d[c].notna() & ~d[c].between(낮은, 높은)
        d = d[~밖]
        로그['이상치'] = n - len(d)
        if 건너뜀:
            로그['울타리가 0이라 건너뛴 열'] = len(건너뜀)
    if missing == '지우기':
        n = len(d); d = d.dropna(subset=features); 로그['빈칸 있는 행'] = n - len(d)
    else:
        로그['채운 빈칸'] = int(d[features].isnull().sum().sum())
        for c in 숫자:
            d[c] = d[c].fillna(d[c].mean())
        for c in 글자:
            if d[c].notna().any():
                d[c] = d[c].fillna(d[c].mode()[0])
    X = pd.get_dummies(d[features], columns=글자, dtype=float).astype(float)   # 글자 → 숫자(원-핫)
    y = d[target].astype(str) if 목적 == '분류' else d[target].astype(float)
    로그['남은 행'] = len(X)
    return X, y, 로그

def 모델_만들기(알고리즘, 목적, 값=None, seed=0):
    if 알고리즘 == 'knn':
        k = int(값 or 5)
        return KNeighborsClassifier(n_neighbors=k) if 목적 == '분류' else KNeighborsRegressor(n_neighbors=k)
    if 알고리즘 == 'tree':
        틀 = DecisionTreeClassifier if 목적 == '분류' else DecisionTreeRegressor
        return 틀(max_depth=int(값 or 3), random_state=seed)
    if 알고리즘 == 'linear':
        if 목적 == '분류':
            raise ValueError('선형 회귀는 숫자를 맞히는 회귀에만 써요. knn이나 tree를 골라요.')
        if 값 is not None:
            raise ValueError("선형 회귀에는 바꿀 하이퍼파라미터가 없어요. 알고리즘='knn'이나 'tree'와 함께 값을 바꿔 보세요.")
        return LinearRegression()
    raise ValueError("알고리즘은 'knn', 'tree', 'linear' 가운데 하나예요.")

def 학습(X, y, 설정, 속성=None, 알고리즘=None, 값=None, 정규화=None, 훈련비율=1.0, 시험비율=None):
    if 속성 is not None:
        모르는 = [f for f in 속성 if f not in 설정['FEATURES']]
        if 모르는:
            raise ValueError(f'FEATURES에 없는 열이에요: {모르는} — 새 열을 쓰려면 1-3의 FEATURES에 더하고 다시 실행해요.')
        if not 속성:
            raise ValueError('속성을 하나 이상 남겨요.')
    열 = [c for c in X.columns if 속성 is None or any(c == f or c.startswith(f + '_') for f in 속성)]
    목적 = 설정['목적']
    알고 = 알고리즘 or 설정['ALGORITHM']
    if 값 is None and 알고 == 설정['ALGORITHM']:
        값 = 설정['PARAM']
    층 = y if 목적 == '분류' and y.value_counts().min() >= 2 else None
    Xtr, Xte, ytr, yte = train_test_split(X[열], y, test_size=시험비율 or 설정['TEST_SIZE'], random_state=설정['SEED'], stratify=층)
    if 훈련비율 < 1.0:
        Xtr = Xtr.sample(frac=훈련비율, random_state=설정['SEED'])
        ytr = ytr.loc[Xtr.index]
    맞춤 = None
    if 설정['SCALE'] if 정규화 is None else 정규화:
        맞춤 = MinMaxScaler().fit(Xtr)                    # 훈련 데이터로만 최솟값·최댓값을 구해요
        Xtr = pd.DataFrame(맞춤.transform(Xtr), index=Xtr.index, columns=열)
        Xte = pd.DataFrame(맞춤.transform(Xte), index=Xte.index, columns=열)
    모델 = 모델_만들기(알고, 목적, 값, 설정['SEED']).fit(Xtr, ytr)
    예측 = 모델.predict(Xte)
    점수 = accuracy_score(yte, 예측) if 목적 == '분류' else mean_absolute_error(yte, 예측)
    return dict(model=모델, scaler=맞춤, cols=열, X_train=Xtr, X_test=Xte, y_train=ytr, y_test=yte, pred=예측, 점수=float(점수))

def 꼭할_실험_설정(실험과제, 설정, 뺄=None):
    알고 = 설정['ALGORITHM']
    if 실험과제 == 실험과제_목록[0]:
        if len(설정['FEATURES']) < 2:
            raise ValueError('입력 속성이 하나뿐이라 뺄 수 없어요. 1-3의 FEATURES에 속성을 하나 더 넣고 다시 실행해요.')
        뺄 = 뺄 or 설정['FEATURES'][0]
        return {'속성': [f for f in 설정['FEATURES'] if f != 뺄]}
    if 실험과제 == 실험과제_목록[1]:
        if 알고 == 'linear':
            return {'알고리즘': 'knn', '값': [1, 5, 15]}
        return {'값': [1, 5, 15] if 알고 == 'knn' else [2, 4, 8]}
    if 실험과제 == 실험과제_목록[2]:
        return {'정규화': not 설정['SCALE']}
    if 실험과제 == 실험과제_목록[3]:
        return {'훈련비율': 0.5}
    후보 = ['knn', 'tree'] if 설정['목적'] == '분류' else ['linear', 'knn', 'tree']
    return {'알고리즘': [a for a in 후보 if a != 알고][0]}

def 원리_설명(결과, 위치, 정답, df_raw, features, target):
    model, X_test, y_train = 결과['model'], 결과['X_test'], 결과['y_train']
    행 = X_test.iloc[[위치]]
    print(f'🔎 테스트 데이터 맨 위에서 {위치 + 1}번째 행 (행 번호 {X_test.index[위치]})')
    print('   원래 값:', {k: (v.item() if hasattr(v, 'item') else v) for k, v in df_raw.loc[X_test.index[위치], features].items()})
    if 결과['scaler'] is not None:
        print('   계산에 쓰는 값(0~1로 크기를 맞춘 값):', {c: round(float(행.iloc[0][c]), 3) for c in 결과['cols'][:12]})
    if 정답:
        print(f'   실제: {결과["y_test"].iloc[위치]} · 예측: {결과["pred"][위치]}')
    if isinstance(model, (KNeighborsClassifier, KNeighborsRegressor)):
        거리, 번호 = model.kneighbors(행)
        print(f'\n📍 가장 가까운 이웃 {len(번호[0])}개')
        for d, j in zip(거리[0], 번호[0]):
            print(f'   행 {y_train.index[j]}: 거리 {d:.3f}' + (f', 정답 {y_train.iloc[j]}' if 정답 else ''))
        if 정답:
            이웃 = y_train.iloc[번호[0]]
            print('   → 투표 결과:', {str(k): int(v) for k, v in 이웃.value_counts().items()} if isinstance(model, KNeighborsClassifier) else f'평균 {이웃.mean():.3f}')
        else:
            print(f'   ✏️ 이웃마다 정답을 df_raw.loc[행 번호, {target!r}]로 찾아 적고, 내 모델이 무엇이라고 답할지 먼저 정해요.')
    elif isinstance(model, (DecisionTreeClassifier, DecisionTreeRegressor)):
        if 정답:
            t = model.tree_
            print('\n🌳 지나간 질문')
            for 노드 in model.decision_path(행).indices:
                if t.children_left[노드] == -1:
                    print(f'   → 잎에 닿음: 예측 {결과["pred"][위치]}')
                else:
                    열이름 = 결과['cols'][t.feature[노드]]
                    기준, 값 = t.threshold[노드], 행.iloc[0][열이름]
                    print(f'   {열이름} ≤ {기준:.3f}? 값 {값:.3f} → {"예" if 값 <= 기준 else "아니오"}')
        else:
            print('\n🌳 내 트리 (위에서부터 질문을 따라 내려가요. 들여쓰기 한 칸이 한 단계예요)')
            글 = export_text(model, feature_names=결과['cols'], decimals=3).splitlines()
            print('\n'.join(글[:60]) + (f'\n   … 모두 {len(글)}줄' if len(글) > 60 else ''))
            print('   ✏️ 이 행의 값으로 질문마다 예/아니오를 정해 잎까지 따라가요.')
    else:
        print('\n📈 직선(평면) 식: 예측 = 절편 + Σ 기울기 × 값')
        print(f'   절편 {model.intercept_:.3f}')
        합 = model.intercept_
        for 열이름, w in zip(결과['cols'], model.coef_):
            값 = 행.iloc[0][열이름]
            합 += w * 값
            print(f'   기울기 {w:.3f} × {열이름}({값:.3f})' + (f' = {w * 값:.3f}' if 정답 else ''))
        if 정답:
            print(f'   합 = {합:.3f}')
        else:
            print('   ✏️ 곱해서 모두 더하면 예측이 나와요. 계산기로 직접 해 봐요.')

명단_글 = '''
10101 김펭귄
10102 이아델
10103 박젠투
'''   # ✏️ 한 줄에 "학번 이름" — 우리 반 명단을 붙여 넣어요

명단 = dict(줄.split(maxsplit=1) for 줄 in 명단_글.strip().splitlines() if 줄.strip())
조건표 = pd.DataFrame([{'학번': h, '이름': n, **나만의_조건(h)} for h, n in 명단.items()])
조건표['확인행'] = 조건표['확인행'] + 1   # 학생 화면처럼 "맨 위에서 몇 번째"로
조건표

## 2. 제출 노트북 한꺼번에 점검
학생들이 낸 노트북(.ipynb)을 구글 드라이브의 한 폴더에 모아요(클래스룸에서 "모든 제출물 다운로드" 뒤 드라이브에 올리기). 학생에게는 **모두 실행 → 저장(Ctrl+S) 뒤에 내라**고 알려 주세요. 실행 결과가 저장되어 있어야 요약코드를 읽을 수 있어요.

- 요약코드의 설정으로 전처리와 학습을 다시 해서 행 수·점수·실험 점수가 학생 노트북과 같은지 봐요. 코드를 몰래 고쳐 점수를 바꾸면 여기서 드러나요.
- 📝 답에서 학생이 더한 글만 떼어 내 빈 답과 비슷한 답(두 학생이 70% 넘게 같은 글)을 찾아요.
- "답 속 숫자"는 학생이 답에 쓴 숫자 가운데 실행 결과에 실제로 나온 숫자의 수예요. 근거가 내 숫자인지 빠르게 가늠할 때 써요.

In [ ]:
FOLDER = '/content/drive/MyDrive/수행평가'   # ✏️ 학생 노트북을 모아 둔 폴더
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Colab이 아니면 FOLDER에 이 컴퓨터의 폴더를 적어요.')

import glob, os, re, difflib
템플릿 = json.loads(r'''{"1-1":"📝 **1-1 답** — 이 데이터를 고른 까닭을 **내 진로나 관심 분야**와 이어서 2~3문장으로 적어요.\n> (예: 나는 ○○ 쪽으로 진학하고 싶은데, 이 데이터로 ~을 미리 알 수 있으면 ~에 도움이 될 것 같아서 골랐다.)\n\n- 고른 데이터(코드):\n- 고른 까닭:","1-2":"📝 **1-2 답** — 빈칸을 채워요.\n\n- **사용 대상** (누가 쓰나요?):\n- **상황** (언제 필요한가요?):\n- **해결할 문제** (무엇을 알고 싶은가요?):\n- **필요한 까닭** (이 프로그램이 없으면 무엇이 불편한가요?):","1-3":"📝 **1-3 답** — 위 결과를 보고 채워요.\n\n- **맞힐 것(정답 y)**:\n- **입력(X)으로 쓸 열과 고른 까닭**: (왜 이 열들이 정답과 관계가 있을까요?)\n- **학습 목적(분류·회귀)과 그렇게 정한 까닭**:\n- **기능 한 줄 설명**: \"사용자가 ______를 넣으면 ______를 알려 주는 프로그램\"\n- **성공 기준(숫자로)**: (예: 정확도 0.85 이상 / 평균 절대 오차 ___ 이하) — 그렇게 정한 까닭:","1-4":"📝 **1-4 답** — **처음 계획** 칸은 1차시에 적고 고치지 않아요. **실제로 한 일** 칸은 수업이 끝날 때마다 적어요(4-3에서 둘을 견줘요).\n\n| 차시 | 처음 계획 (1차시에 적고 고치지 않아요) | 실제로 한 일 (차시마다 적어요) |\n|---|---|---|\n| 1 | (예: 데이터 고르기, ① 1-1 ~ 1-3) |  |\n| 2 |  |  |\n| 3 |  |  |\n| 4 |  |  |\n| 5 |  |  |","2-1":"📝 **2-1 답** — 내 결과의 숫자로 적어요.\n- 결측치가 있는 열과 개수:\n- 이상치가 있는 열과 개수, 그 값이 **측정 실수**일지 **진짜 드문 값**일지에 대한 내 생각:","2-2":"📝 **2-2 답**\n- 고른 결측치 처리와 까닭:\n- 고른 이상치 처리와 까닭:\n- 정규화를 했나요? 까닭:\n- 전처리 기록의 숫자로: 처음 ___행 → 남은 ___행, 빠진 까닭:","2-4":"📝 **2-4 답**\n- 넣은 값과 예측 결과:\n- 이 예측이 그럴듯한가요? 데이터(평균값, 비슷한 행 등)를 근거로 판단해요:","2-5":"📝 **2-5 답** — 고른 알고리즘(ALGORITHM)과 그 까닭. 수업에서 배운 원리(가까운 이웃의 투표 · 예/아니오 질문 · 직선)와 **내 데이터의 특징**을 이어서 적어요.","3-1":"📝 **3-1 답**\n- 내 모델의 성능(숫자):\n- \"찍기\"보다 얼마나 나은가요?(숫자):\n- 1-3에서 정한 **성공 기준**을 넘었나요?:\n- (분류) 혼동 행렬에서 가장 많이 헷갈린 짝:","3-2":"📝 **3-2 답** — 틀린 예 2개(틀린 예가 2개보다 적으면 확신이 가장 낮은 예)를 골라 **행 번호**를 적고, 왜 틀렸을지(또는 아슬아슬했을지) **값을 근거로** 설명해요.\n- 행 번호 ___ : (예: 부리길이 46.1로 턱끈 평균 48.8보다 젠투 평균 47.5에 더 가까워서 …)\n- 행 번호 ___ :","3-3":"📝 **3-3 답** — 실험마다 한 줄씩. **예상과 결과가 다르면 그 까닭**을 꼭 적어요(다른 결과가 오히려 좋은 분석거리예요).\n\n| 번호 | 가설 (실행 전 예상) | 바꾼 것 | 결과 (숫자) | 예상과 같았나? 왜 그렇게 됐을까? |\n|---|---|---|---|---|\n| 1 |  |  |  |  |\n| 2 |  |  |  |  |\n| 3 |  |  |  |  |","3-4":"📝 **3-4 답** — 3-1 ~ 3-3의 **숫자를 근거로** 적어요. 개선 방안은 \"무엇을 · 어떻게 · 기대 효과\"까지.\n\n- **한계 1 (데이터)**: → 개선 방안:\n- **한계 2 (모델·알고리즘)**: → 개선 방안:\n- **한계 3 (윤리·공정성)**: 데이터의 치우침(편향), 개인정보, 이 프로그램이 틀렸을 때 누가 어떤 피해를 볼 수 있는지 → 개선 방안:","4-1":"📝 **4-1 답** — 위 그래프 두 개가 각각 무엇을 보여 주는지, 가장 눈에 띄는 점을 숫자와 함께 2문장씩 적어요.","4-1(나)":"📝 **4-1(나) 답** — 무엇을 만들었고, 왜 그 사용 대상에게 알맞은가요?","4-2(가)":"📝 **4-2(가) 답** — 답을 보기 전에: 내 모델이 이 행을 무엇이라고 답할지, 어떤 과정(지나는 질문 · 이웃의 투표 · 식의 계산)으로 그렇게 되는지 적어요.","4-2(나)":"📝 **4-2(나) 답** — (가)에서 내가 따라간 것과 (나)가 같았나요? 다르면 어디서 갈렸고 왜 그런지, 위 출력의 **숫자를 써서** 설명해요.","4-3":"📝 **4-3 답**\n- 가장 어려웠던 순간과 해결한 방법:\n- 1-4의 처음 계획과 실제로 한 일이 달라진 점과 그 까닭:\n- 다시 한다면 바꿀 것:","4-4":"📝 **4-4 답** — AI 챗봇을 쓴 만큼 줄을 늘려요. 쓰지 않았다면 \"쓰지 않음\"이라고 적어요.\n\n| 질문 번호 | AI에게 물은 것 (그대로 붙여 넣기) | AI 답을 어떻게 썼나 (그대로 · 고쳐서 · 안 씀) | 내가 확인한 방법 (실행 결과 · 자료 · 선생님) |\n|---|---|---|---|\n|  |  |  |  |\n\n- **AI를 썼다면** — AI가 틀렸거나 내 데이터에 맞지 않았던 경우 한 가지와 내가 고친 방법:\n- **AI를 쓰지 않았다면** — 내가 스스로 찾아 고친 오류 한 가지와 확인한 방법:"}''')

def _글(셀):
    s = 셀.get('source', '')
    return ''.join(s) if isinstance(s, list) else s

def _출력(셀):
    글 = []
    for o in 셀.get('outputs', []):
        t = o.get('text') or o.get('data', {}).get('text/plain', '')
        글.append(''.join(t) if isinstance(t, list) else str(t))
    return '\n'.join(글)

def _더한글(키, 글):
    원래 = set(줄.strip() for 줄 in 템플릿.get(키, '').splitlines())
    return '\n'.join(줄 for 줄 in 글.splitlines() if 줄.strip() and 줄.strip() not in 원래)

def _숫자들(글):
    return re.findall(r'-?\d+(?:\.\d+)?', 글)

def 노트북_읽기(경로):
    nb = json.load(open(경로, encoding='utf-8'))
    답 = {}
    출력 = ''
    for 셀 in nb.get('cells', []):
        if 셀.get('cell_type') == 'markdown':
            m = re.match(r'📝 \*\*(.+?) 답\*\*', _글(셀))
            if m:
                답[m.group(1)] = _더한글(m.group(1), _글(셀))
        elif 셀.get('cell_type') == 'code':
            출력 += _출력(셀) + '\n'
    m = re.search(r'^요약코드: (\{.*\})\s*$', 출력, re.M)
    지문 = re.search(r'🔏 지문: ([0-9a-f]{8})', 출력)
    return dict(파일=os.path.basename(경로), 요약=json.loads(m.group(1)) if m else None, 지문=지문.group(1) if 지문 else '', 답=답, 출력=출력)

def 다시_계산(요약):
    df = 데이터_불러오기(요약['DATASET'], 요약.get('MY_CSV', ''))
    X, y, 기록 = 전처리(df, 요약['FEATURES'], 요약['TARGET'], 요약['목적'], 요약['MISSING'], 요약['OUTLIER'])
    설정 = {k: 요약[k] for k in ('목적', 'FEATURES', 'SEED', 'TEST_SIZE', 'SCALE', 'ALGORITHM', 'PARAM')}
    결과 = 학습(X, y, 설정)
    실험점수 = []
    for e in 요약['실험']:
        바꿀것 = e['바꿀것']
        if isinstance(바꿀것.get('값'), list):
            실험점수.append([round(학습(X, y, 설정, **{**바꿀것, '값': v})['점수'], 4) for v in 바꿀것['값']])
        else:
            실험점수.append(round(학습(X, y, 설정, **바꿀것)['점수'], 4))
    return dict(df=df, X=X, y=y, 설정=설정, 결과=결과, 행수=len(X), 점수=round(결과['점수'], 4), 실험점수=실험점수)

def 꼭할_맞나(요약):
    조건_ = 나만의_조건(요약['학번'])
    꼭 = [e for e in 요약['실험'] if e.get('꼭할')]
    if not 꼭:
        return False
    설정 = {k: 요약[k] for k in ('목적', 'FEATURES', 'SEED', 'TEST_SIZE', 'SCALE', 'ALGORITHM', 'PARAM')}
    기대 = 꼭할_실험_설정(조건_['실험과제'], 설정)
    바꿀것 = 꼭[0]['바꿀것']
    if '속성' in 기대:
        return '속성' in 바꿀것 and len(바꿀것['속성']) == len(요약['FEATURES']) - 1
    return 바꿀것 == 기대

점검 = []
for 경로 in sorted(glob.glob(os.path.join(FOLDER, '*.ipynb'))):
    r = 노트북_읽기(경로)
    s = r['요약']
    줄 = {'파일': r['파일'], '학번': s['학번'] if s else '', '이름': s['이름'] if s else '', '⚠️': []}
    if not s:
        줄['⚠️'].append('요약코드 없음(모두 실행·저장 뒤 다시 내기)')
        점검.append(dict(줄, 읽은것=r)); continue
    if s['학번'] not in 명단:
        줄['⚠️'].append('명단에 없는 학번')
    elif 명단[s['학번']] != s['이름']:
        줄['⚠️'].append(f'이름이 명단과 다름({명단[s["학번"]]})')
    if 지문_만들기(s) != r['지문']:
        줄['⚠️'].append('지문이 요약코드와 다름(출력을 손으로 고침?)')
    기대 = 나만의_조건(s['학번'])
    if (s['SEED'], s['TEST_SIZE']) != (기대['SEED'], 기대['TEST_SIZE']):
        줄['⚠️'].append('SEED·TEST_SIZE가 학번의 조건과 다름')
    if not 꼭할_맞나(s):
        줄['⚠️'].append('꼭 할 실험을 하지 않았거나 다르게 함')
    if len(s['실험']) < 3:
        줄['⚠️'].append(f'실험 {len(s["실험"])}개')
    try:
        재 = 다시_계산(s)
        같음 = 재['행수'] == s['행수'] and abs(재['점수'] - s['점수']) < 1e-4 and 재['실험점수'] == [e['점수'] for e in s['실험']]
        줄['다시 계산'] = '✅ 같음' if 같음 else f'⚠️ 다름(행 {재["행수"]}, 점수 {재["점수"]}, 실험 {재["실험점수"]})'
        if not 같음:
            줄['⚠️'].append('다시 계산한 결과가 다름(코드나 결과를 고침?)')
        r['다시'] = 재
    except Exception as 오류:
        줄['다시 계산'] = f'건너뜀({type(오류).__name__})'
    빈답 = [k for k in 템플릿 if not r['답'].get(k, '').strip()]
    줄['빈 답'] = len(빈답)
    if 빈답:
        줄['⚠️'].append('빈 답: ' + ', '.join(빈답))
    답숫자 = [n for 글 in r['답'].values() for n in _숫자들(글)]
    출력숫자 = set(_숫자들(r['출력']))
    맞는숫자 = [n for n in 답숫자 if n in 출력숫자 or any(o.startswith(n) for o in 출력숫자 if '.' in n)]
    줄['답 속 숫자(출력에 있는 것/전체)'] = f'{len(맞는숫자)}/{len(답숫자)}'
    점검.append(dict(줄, 읽은것=r))

# 비슷한 답 — 학생 둘이 더한 글이 70% 넘게 같으면 표시해요
for i in range(len(점검)):
    for j in range(i + 1, len(점검)):
        a, b = 점검[i]['읽은것']['답'], 점검[j]['읽은것']['답']
        for 키 in 템플릿:
            x, y_ = a.get(키, ''), b.get(키, '')
            if min(len(x), len(y_)) >= 30 and difflib.SequenceMatcher(None, x, y_).ratio() > 0.7:
                점검[i]['⚠️'].append(f'{키} 답이 {점검[j]["학번"]}와 비슷함')
                점검[j]['⚠️'].append(f'{키} 답이 {점검[i]["학번"]}와 비슷함')
학번들 = [z['학번'] for z in 점검]
for z in 점검:
    if z['학번'] and 학번들.count(z['학번']) > 1:
        z['⚠️'].append('같은 학번 노트북이 둘 이상')

점검표 = pd.DataFrame([{k: (' / '.join(v) if k == '⚠️' else v) for k, v in z.items() if k != '읽은것'} for z in 점검])
if len(점검표):
    점검표.to_csv('점검표.csv', index=False, encoding='utf-8-sig')
    print(f'📋 노트북 {len(점검표)}개를 점검했어요. ⚠️가 있는 학생 {sum(1 for z in 점검 if z["⚠️"])}명 — 점검표.csv(왼쪽 📁 파일)에도 저장했어요.')
else:
    print('FOLDER에 .ipynb 파일이 없어요. 폴더 위치를 확인해요.')
점검표

## 3. 현장 확인 설명지와 답 열쇠
마지막 차시에 AI 챗봇 없이 10분, **학생이 자기 노트북만 보며** 평가 요소마다 한 문제씩 답해요. ④의 행은 비밀 낱말로 학생마다 다르게 골라서, 미리 알 수 없어요.

- 학생마다 다른 숫자가 들어간 설명지를 만들어요. ④는 비밀 낱말로 고른 행이라 학생이 미리 준비할 수 없어요.
- 답 열쇠에는 ④ 행의 정답·예측과 지나간 질문(또는 이웃의 투표, 식의 계산)이 들어 있어요.

In [ ]:
비밀 = '✏️ 선생님만 아는 낱말'   # ✏️ 해마다 바꿔요 — 학생마다의 ④ 행을 고를 때 써요

def 현장_행(학번, 테스트수, 확인행):
    h = int(hashlib.sha256((비밀 + ':' + str(학번)).encode('utf-8')).hexdigest(), 16)
    후보 = [i for i in range(min(테스트수, 12)) if i != 확인행]
    return 후보[h % len(후보)]

import io, contextlib
설명지, 열쇠 = [], []
for z in 점검:
    s, 재 = z['읽은것']['요약'], z['읽은것'].get('다시')
    if not s or not 재:
        continue
    위치 = 현장_행(s['학번'], len(재['결과']['X_test']), s['확인행'])
    꼭 = next((e for e in s['실험'] if e.get('꼭할')), None)
    설명지.append('\n'.join([
        f'## 현장 설명지 — {s["학번"]} {s["이름"]}  (AI 챗봇·검색·대화 없이, 내 노트북만 보며 10분)',
        f'① 1-3에 적은 성공 기준을 그 숫자로 정한 까닭을 두 문장으로 적어요.',
        f'② 전처리 전 {s["처음행수"]}행이 {s["행수"]}행이 되었어요. 전처리 기록의 숫자로 그 까닭을 설명해요.',
        f'③ 꼭 할 실험({꼭["바꿀것"] if 꼭 else "—"})의 결과({꼭["점수"] if 꼭 else "—"})를 처음 가설과 견주어 설명해요.',
        f'④ 노트북 맨 아래 현장_행에 {위치 + 1}을 적고 실행해요. 그 행을 내 모델이 무엇이라고 답할지, 어떤 과정으로 그렇게 되는지 손으로 따라가 적어요.',
        '']))
    글 = io.StringIO()
    with contextlib.redirect_stdout(글):
        원리_설명(재['결과'], 위치, True, 재['df'], s['FEATURES'], s['TARGET'])
    열쇠.append(f'## 답 열쇠 — {s["학번"]} {s["이름"]} · ④ 테스트 맨 위에서 {위치 + 1}번째 행\n' + 글.getvalue())

open('현장설명지.md', 'w', encoding='utf-8').write('\n'.join(설명지))
open('현장답열쇠.md', 'w', encoding='utf-8').write('\n'.join(열쇠))
print(f'📝 현장 설명지 {len(설명지)}장 · 답 열쇠 {len(열쇠)}장을 만들었어요 (왼쪽 📁 파일에서 내려받아 인쇄).')
print((설명지[0] if 설명지 else ''))

## 4. 점수 합계 — 현장 확인으로 상한 두기
노트북에서 정한 수준과 현장 확인 결과를 적어요. 현장 확인이 **○면 노트북 수준 그대로**, **△면 중간 수준까지**, **×면 가장 낮은 수준**이에요.

- 노트북 수준(① 30·20·10, ② 30·20·10, ③ 25·15·5, ④ 15·10·5)과 현장 확인(○·△·×)을 적어요. 가장 낮은 수준의 합이 기본점수 30점이에요.
- 허용되지 않은 점수나 비어 있는 요소가 있으면 오류로 알려 줘요.

In [ ]:
# ✏️ 한 줄에 학생 한 명: 학번, 이름, 노트북 수준 ①②③④, 현장 확인 ①②③④(○·△·×), 메모
# 낼 뜻이 없어 내지 않은 학생은 노트북 수준을 모두 None으로 두면 20점(미응시)이 돼요.
채점 = [
    ['10101', '김펭귄', 30, 20, 25, 15, '○', '○', '△', '○', ''],
    ['10102', '이아델', None, None, None, None, '', '', '', '', '미제출'],
]

수준 = [(30, 20, 10), (30, 20, 10), (25, 15, 5), (15, 10, 5)]   # 요소마다 (상, 중, 하)

def 최종(점수, 현장, 단계):
    상, 중, 하 = 단계
    if 현장 == '×':
        return 하
    if 현장 == '△':
        return min(점수, 중)
    return 점수

표 = []
for 줄 in 채점:
    학번_, 이름_, 노트북, 현장 = 줄[0], 줄[1], 줄[2:6], 줄[6:10]
    if all(s is None for s in 노트북):
        표.append([학번_, 이름_, *[None] * 4, 20, 줄[10]]); continue
    if any(s is None for s in 노트북):
        raise ValueError(f'{학번_}: 아직 비어 있는 요소가 있어요')
    for s, 단계 in zip(노트북, 수준):
        if s not in 단계:
            raise ValueError(f'{학번_}: {s}는 허용된 점수가 아니에요 {단계}')
    끝 = [최종(s, h, 단계) for s, h, 단계 in zip(노트북, 현장, 수준)]
    표.append([학번_, 이름_, *끝, sum(끝), 줄[10]])

채점표 = pd.DataFrame(표, columns=['학번', '이름', '①문제정의', '②데이터·구현', '③성능·개선', '④표현·설명', '합계', '메모'])
채점표.to_csv('채점표.csv', index=False, encoding='utf-8-sig')
print('💾 채점표.csv를 만들었어요 (왼쪽 📁 파일에서 내려받기)')
채점표